# Bengali Pitha Image Classification Using Xception Deep Features

## Overview

Pitha are traditional Bengali rice-based cakes, and many varieties look very similar to one another, so recognizing them from photographs is a fine-grained image classification problem. This notebook implements a complete and reproducible classification pipeline for a curated dataset of Bengali pitha images (one folder per class). A frozen, ImageNet-pretrained **Xception** network serves as a fixed feature extractor, the extracted features are compressed with **Principal Component Analysis (PCA)**, and four classical machine-learning classifiers are trained, compared and evaluated.

### Pipeline

1. **Test set selected by hand** (read unchanged from its own folder, never augmented); the remaining curated images are split by class into training and validation subsets (89 / 11).
2. **Augmentation of the training subset only**, performed after the split.
3. **Feature extraction** with a frozen Xception backbone (global average pooling, 2048-dimensional vectors).
4. **Dimensionality reduction** with PCA (256 components), fitted on training features only.
5. **Classifier training** of an SVM, Logistic Regression, Random Forest and XGBoost model.
6. **Model selection** based on validation accuracy.
7. **Final evaluation** of the selected model on the held-out test set (metrics, confusion matrix, ROC curves) and **error analysis** of the misclassified images.

### Classifiers

| Classifier | Implementation | Key settings | Characteristics |
|---|---|---|---|
| SVM (RBF kernel) | `sklearn.svm.SVC` | `C=1.0`, `gamma=0.01`, probability estimates enabled | Kernel-based maximum-margin classifier, well suited to compact dense features |
| Logistic Regression | `sklearn.linear_model.LogisticRegression` | `max_iter=2000`, multinomial (softmax) formulation | Linear probabilistic baseline |
| Random Forest | `sklearn.ensemble.RandomForestClassifier` | 500 trees | Ensemble of bagged decision trees |
| XGBoost | `xgboost.XGBClassifier` | 500 boosting rounds, `multi:softprob` objective | Gradient-boosted decision trees |

### Evaluation Protocol

The following safeguards prevent information from the evaluation data from influencing training or model selection:

- **Test set selected by hand.** The test images live in their own folder, are never augmented, and any byte-identical copy in the training pool is removed before splitting.
- **Split before augmentation.** No augmented copy of a validation or test image can appear in the training set.
- **PCA fitted on training data only.** Validation and test features are only transformed, never used to fit the projection.
- **Model selection on validation accuracy.** The test set is never consulted while choosing the classifier.
- **Single test evaluation.** The selected model is evaluated once on the untouched test set; this is the headline result.
- **Traceability.** Filenames are stored with every feature vector so each prediction can be traced back to the source image.

### Requirements

The notebook runs on **Google Colab**, which already provides every required package (TensorFlow, scikit-learn, XGBoost, OpenCV, Pillow, pandas, Matplotlib, seaborn). Only the data have to be on Google Drive:

```
MyDrive/
├── clean_data/<class_name>/*.jpg           training + validation pool
├── split_data_v2/test/<class_name>/*.jpg   test set, selected by hand
└── pitha_outputs/                          created: a copy of every result zip
```

Everything else is created on the Colab disk while the notebook runs:

```
/content/
├── pitha_scripts/            pitha_preprocessing.py, split_and_augment.py, pitha_metadata.json
├── split_data/               generated: train / val / test folders and split manifest
└── outputs/model_xception/  generated: models, figures/ and tables/ -> model_xception_results.zip
```

### Outputs

Every figure and table is displayed **and** written to disk by the cell that produces it:

- `model_xception/` holds the trained model, PCA transform, label encoder and extracted features.
- `model_xception/figures/` holds all figures as PNG files (300 dpi).
- `model_xception/tables/` holds all result tables as CSV files.

At the end, everything is packed into `model_xception_results.zip`, which is downloaded to your computer and also saved to `pitha_outputs/` on Google Drive (Section 12).

A complete list of generated files is given in Section 11.

## 0. Google Colab Setup

This notebook runs on **Google Colab** with the data on **Google Drive**.

**Before running**

1. Select a GPU runtime: *Runtime → Change runtime type → T4 GPU*. Feature extraction and XGBoost use the GPU; the notebook also runs on a CPU, only more slowly.
2. Check that Google Drive contains the two data folders, each with one sub-folder per class and identical class-folder names in both:

```
MyDrive/
├── clean_data/<class_name>/*.jpg           training + validation pool
└── split_data_v2/test/<class_name>/*.jpg   test set, selected by hand
```

   If they sit inside a sub-folder of *My Drive*, set `DRIVE_DATA_ROOT` in the next cell — or leave it, and the cell searches Drive for them.
3. Run all cells in order (*Runtime → Run all*) and approve Google Drive access when asked.

The cells below mount Drive, check the data folders, and write the three helper files the pipeline needs (`pitha_preprocessing.py`, `split_and_augment.py`, `pitha_metadata.json`) to the Colab disk, so nothing else has to be uploaded. When the notebook finishes, all results are downloaded as one zip file (Section 12).

In [ ]:
# ---------------------------------------------------------------- Google Drive
import os, sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ML dataset')

CLEAN_DATA_DIR   = DRIVE_DATA_ROOT / 'clean_data'               # training + validation pool
TEST_DATA_DIR    = DRIVE_DATA_ROOT / 'split_data_v2' / 'test'   # test set, selected by hand
DRIVE_OUTPUT_DIR = DRIVE_DATA_ROOT / 'pitha_outputs'            # a copy of every result zip is saved here

# ---------------------------------------------------------------- quick check of the inputs
IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}


def images_per_class(folder):
    # number of image files in each class sub-folder (hidden folders are ignored)
    return {d.name: sum(1 for f in d.iterdir() if f.suffix.lower() in IMG_EXT)
            for d in sorted(folder.iterdir()) if d.is_dir() and not d.name.startswith('.')}


pool_counts = images_per_class(CLEAN_DATA_DIR)
test_counts = images_per_class(TEST_DATA_DIR)
print(f'Data folder : {DRIVE_DATA_ROOT}')
print(f'clean_data  : {len(pool_counts):2d} classes, {sum(pool_counts.values())} images (training + validation pool)')
print(f'test        : {len(test_counts):2d} classes, {sum(test_counts.values())} images (selected by hand)')
print(f'total       : {sum(pool_counts.values()) + sum(test_counts.values())} images')



# ---------------------------------------------------------------- local folder for the helper files
Path('/content/pitha_scripts').mkdir(parents=True, exist_ok=True)
print('\nNext: run the three cells below to write the helper files.')

In [ ]:
%%writefile /content/pitha_scripts/pitha_preprocessing.py
"""
Pitha Preprocessing & Curation Module (Pipeline V2)
---------------------------------------------------
Reusable image preprocessing functions used across:
  - Dataset splitting & standardization (split_and_augment.py)
  - Inference on new camera photos (classify_new_pitha.py)

Key Features:
  1. EXIF Auto-Orientation: Resolves phone camera orientation.
  2. Non-Destructive: Keeps authentic dough and crust food textures intact.
  3. Aspect-Preserving Reflection Padding: Scales and pads images to 224×224
     without stretching or distorting pitha geometry.
  4. Laplacian Blur Quality Audit: Measures edge variance to flag blurry images.
"""

from pathlib import Path
from PIL import Image, ImageOps
import cv2
import numpy as np

VALID_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
MIN_RESOLUTION = 140
BLUR_THRESHOLD = 20.0


def load_and_orient_image(img_path):
    """
    Loads image using PIL, resolves EXIF camera rotation,
    and converts color space to 3-channel RGB.
    Returns PIL Image or None on failure.
    """
    try:
        with Image.open(img_path) as img:
            img = ImageOps.exif_transpose(img)
            img = img.convert("RGB")
            return img
    except Exception:
        return None


def evaluate_quality(pil_img, min_res=MIN_RESOLUTION, blur_threshold=BLUR_THRESHOLD):
    """
    Checks resolution boundaries and computes Laplacian sharpness variance.
    Returns (passes_quality: bool, score: float, status: str).
    """
    w, h = pil_img.size
    if w < min_res or h < min_res:
        return False, 0.0, f"Resolution too low ({w}x{h})"

    np_img = np.array(pil_img)
    gray = cv2.cvtColor(np_img, cv2.COLOR_RGB2GRAY)
    variance = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    is_blurry = variance < blur_threshold
    status = "Blurry" if is_blurry else "Sharp"
    return not is_blurry, variance, status


def resize_with_pad(img_np, size=224):
    """
    Resize an RGB or BGR numpy array preserving aspect ratio,
    then pad to square (size × size).
    Uses BORDER_REFLECT_101 to avoid harsh black borders.
    """
    h, w = img_np.shape[:2]
    if h == 0 or w == 0:
        raise ValueError(f"Cannot resize an empty image of shape {img_np.shape}")

    scale = size / max(h, w)
    # A very elongated photo (e.g. 5000x10) scales its short side down to 0,
    # which makes cv2.resize raise. Clamp to at least one pixel.
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))

    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(img_np, (new_w, new_h), interpolation=interp)

    top = (size - new_h) // 2
    bottom = size - new_h - top
    left = (size - new_w) // 2
    right = size - new_w - left

    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right, borderType=cv2.BORDER_REFLECT_101
    )
    return padded


def preprocess_for_model(image_path, target_size=224):
    """
    Preprocesses a new single image file for EfficientNetV2:
      1. Loads with EXIF auto-orientation.
      2. Audits sharpness score.
      3. Resizes with aspect-preserving reflection padding to target_size.
    Returns (img_rgb_np: np.ndarray, quality_info: dict) or (None, None).
    """
    pil_img = load_and_orient_image(image_path)
    if pil_img is None:
        return None, None

    passes_quality, score, status = evaluate_quality(pil_img)
    np_rgb = np.array(pil_img)
    model_input = resize_with_pad(np_rgb, size=target_size)

    quality_info = {
        "sharpness_score": score,
        "status": status,
        "original_size": pil_img.size,
    }
    return model_input, quality_info


In [ ]:
%%writefile /content/pitha_scripts/split_and_augment.py
"""
Dataset Stratified Splitting & Selective Augmentation (Pipeline V2.1)
---------------------------------------------------------------------
Splits original curated pitha images into Train and Validation (and, optionally,
Test) sets BEFORE applying data augmentation.

Two modes:

  A. Fixed test set (--test-input given)      <- used by the current notebook
       * The test set is a separate, pre-sorted folder (<class>/<images>).
       * clean_data/ is split into Train and Validation only
         (default 85% / 15%; the two ratios must sum to 1.0).
       * Every clean_data image that is byte-identical to a test image is REMOVED
         from the train/val pool, so the test set cannot leak into training.
       * Test images are only standardized (EXIF orientation + 224x224 padding),
         never augmented.

  B. Three-way split (no --test-input)         <- original behaviour
       * clean_data/ is split into Train / Validation / Test (default 70/15/15).

Common to both modes:
  1. The split is performed on pure original images, then ONLY the training
     originals are augmented (num-augments copies each, named <stem>_aug<k>.jpg).
  2. Validation and test images stay unaugmented.
  3. Byte-identical duplicates inside the pool are collapsed to one image before
     splitting, so a duplicate cannot land in both train and validation.
  4. Output stems are made safe for the notebook's group logic: '_aug' is
     reserved for augmented copies (an original containing it is renamed to
     '-aug'), and stems are unique within a class even when two source files
     differ only by extension (every output is written as .jpg).
  5. All images are standardized to target-size x target-size using
     aspect-ratio preserving reflection padding.

The script refuses to write into an output folder that already contains a split,
so a new run can never silently mix files with an older one.

Outputs:
  output_dir/
  |-- train/             (originals + augmented copies)
  |-- val/               (pure originals)
  |-- test/              (pure originals)
  `-- split_manifest.json (audit log: assignments, removed duplicates, config)
"""

import sys
import json
import hashlib
import argparse
from pathlib import Path
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

from pitha_preprocessing import (
    VALID_EXT,
    load_and_orient_image,
    resize_with_pad,
)


def random_horizontal_flip(img, p=0.5):
    """Flip image horizontally with probability p."""
    if np.random.random() < p:
        return cv2.flip(img, 1)
    return img.copy()


def random_rotation(img, max_angle=15):
    """Rotate image by random angle in [-max_angle, +max_angle]."""
    h, w = img.shape[:2]
    angle = np.random.uniform(-max_angle, max_angle)
    matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(
        img, matrix, (w, h), borderMode=cv2.BORDER_REFLECT_101
    )
    return rotated


def random_color_jitter(img, brightness=0.2, contrast=0.2, saturation=0.2):
    """Randomly adjust brightness, contrast, and saturation."""
    result = img.astype(np.float32)

    # Brightness
    b_factor = 1.0 + np.random.uniform(-brightness, brightness)
    result = result * b_factor

    # Contrast
    c_factor = 1.0 + np.random.uniform(-contrast, contrast)
    mean = np.mean(result)
    result = (result - mean) * c_factor + mean

    # Saturation
    hsv = cv2.cvtColor(np.clip(result, 0, 255).astype(np.uint8),
                       cv2.COLOR_BGR2HSV).astype(np.float32)
    s_factor = 1.0 + np.random.uniform(-saturation, saturation)
    hsv[:, :, 1] = hsv[:, :, 1] * s_factor
    hsv = np.clip(hsv, 0, 255).astype(np.uint8)
    result = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return result


def random_crop_resize(img, min_area_ratio=0.85):
    """Random crop between min_area_ratio and 100%, then resize back."""
    h, w = img.shape[:2]
    area_ratio = np.random.uniform(min_area_ratio, 1.0)
    scale = np.sqrt(area_ratio)

    new_h = int(h * scale)
    new_w = int(w * scale)

    top = np.random.randint(0, h - new_h + 1)
    left = np.random.randint(0, w - new_w + 1)

    cropped = img[top:top + new_h, left:left + new_w]
    resized = cv2.resize(cropped, (w, h), interpolation=cv2.INTER_CUBIC)
    return resized


def add_gaussian_noise(img, sigma=0.01):
    """Add small Gaussian noise."""
    noise = np.random.normal(0, sigma * 255, img.shape).astype(np.float32)
    noisy = img.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)


def augment_image(img):
    """Apply stochastic augmentation pipeline."""
    aug = random_horizontal_flip(img)
    aug = random_rotation(aug)
    aug = random_color_jitter(aug)
    aug = random_crop_resize(aug)
    aug = add_gaussian_noise(aug)
    return aug


def process_image_file(img_path, target_size=224):
    """
    Load with EXIF orientation and pad to a target_size square.
    Returns a BGR numpy array, or None if the file cannot be processed.

    Failures are swallowed deliberately: this runs over thousands of files and a
    single corrupt or degenerate photo should not abort a 40-minute job.
    """
    try:
        pil_img = load_and_orient_image(img_path)
        if pil_img is None:
            return None
        img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        return resize_with_pad(img_bgr, size=target_size)
    except Exception as exc:
        print(f"  [warn] Skipping unreadable image {img_path.name}: {exc}")
        return None


def write_jpg(path, img_bgr):
    """Write a BGR image as a quality-95 JPEG."""
    cv2.imwrite(str(path), img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])


def file_md5(path):
    """MD5 of the raw file bytes; equal hashes mean byte-identical images."""
    return hashlib.md5(path.read_bytes()).hexdigest()


def list_images(class_dir):
    """Sorted image files directly inside a class folder."""
    return sorted(f for f in class_dir.iterdir() if f.suffix.lower() in VALID_EXT)


def unique_stem(stem, digest, used):
    """
    Output stem that is safe for the notebook's group logic.

    '_aug' is reserved for augmented copies, so it is replaced in original names.
    Stems are kept unique (case-insensitively) within a class: every output is a
    .jpg, so '12.jpg' and '12.png' would otherwise overwrite each other and
    share one group id.
    """
    stem = stem.replace("_aug", "-aug")
    if stem.lower() in used:
        stem = f"{stem}-{digest[:6]}"
    used.add(stem.lower())
    return stem


def die(message):
    """Report a fatal problem and stop with a non-zero exit code."""
    print(f"Error: {message}")
    sys.exit(1)


def main():
    parser = argparse.ArgumentParser(
        description="Stratified Train/Val(/Test) Split with Selective Augmentation"
    )
    parser.add_argument("--input", required=True,
                        help="Path to clean_data directory containing curated class folders")
    parser.add_argument("--output", required=True,
                        help="Path to output split_data directory (will contain train/, val/, test/)")
    parser.add_argument("--test-input", default=None,
                        help="Path to a pre-sorted test folder (<class>/<images>). When given, "
                             "--input is split into train/val only and this folder becomes test/")
    parser.add_argument("--train-ratio", type=float, default=None,
                        help="Proportion of originals for training "
                             "(default 0.85 with --test-input, otherwise 0.70)")
    parser.add_argument("--val-ratio", type=float, default=None,
                        help="Proportion of originals for validation (default 0.15)")
    parser.add_argument("--test-ratio", type=float, default=None,
                        help="Proportion of originals for testing (default 0.15; "
                             "not allowed together with --test-input)")
    parser.add_argument("--num-augments", type=int, default=5,
                        help="Number of augmented copies per training image (default 5)")
    parser.add_argument("--target-size", type=int, default=224,
                        help="Square resolution for padded images (default 224)")
    parser.add_argument("--seed", type=int, default=42,
                        help="Random seed for reproducible split (default 42)")
    args = parser.parse_args()

    # Resolve the ratios for the chosen mode
    fixed_test = args.test_input is not None
    if fixed_test:
        if args.test_ratio is not None:
            die("--test-ratio cannot be combined with --test-input "
                "(the test set is the folder you supplied).")
        train_ratio = 0.85 if args.train_ratio is None else args.train_ratio
        val_ratio = 0.15 if args.val_ratio is None else args.val_ratio
        test_ratio = 0.0
    else:
        train_ratio = 0.70 if args.train_ratio is None else args.train_ratio
        val_ratio = 0.15 if args.val_ratio is None else args.val_ratio
        test_ratio = 0.15 if args.test_ratio is None else args.test_ratio

    total_ratio = train_ratio + val_ratio + test_ratio
    if abs(total_ratio - 1.0) > 1e-4:
        die(f"Split ratios must sum to 1.0 (got {total_ratio})")

    # Seed the augmentation RNG too, not just the split, so a rerun with the
    # same seed reproduces the exact same augmented training images.
    np.random.seed(args.seed)

    input_dir = Path(args.input)
    output_dir = Path(args.output)
    test_input = Path(args.test_input) if fixed_test else None

    if not input_dir.exists():
        die(f"Input directory not found: {input_dir}")
    if fixed_test and not test_input.exists():
        die(f"Test directory not found: {test_input}")

    # Refuse to mix a new split into an old one
    for existing in (output_dir / "train", output_dir / "val", output_dir / "test",
                     output_dir / "split_manifest.json"):
        if existing.exists() and (existing.is_file() or any(existing.iterdir())):
            die(f"{existing} already exists. Choose a new --output folder or "
                f"delete/rename the old split first.")

    # Discover classes
    class_dirs = sorted([d for d in input_dir.iterdir() if d.is_dir()])
    if not class_dirs:
        die(f"No class subdirectories found in {input_dir}")
    class_names = {d.name for d in class_dirs}

    # Fixed test set: check class names and hash every test image
    test_class_dirs = []
    test_hashes = {}
    if fixed_test:
        test_class_dirs = sorted([d for d in test_input.iterdir() if d.is_dir()])
        if not test_class_dirs:
            die(f"No class subdirectories found in {test_input}")
        unknown = sorted({d.name for d in test_class_dirs} - class_names)
        if unknown:
            die("Test classes not found in the training data (folder names must match "
                f"exactly): {unknown}")
        for d in test_class_dirs:
            for f in list_images(d):
                test_hashes.setdefault(file_md5(f), []).append(f"{d.name}/{f.name}")

    print("=" * 80)
    print("DATASET STRATIFIED SPLIT & SELECTIVE AUGMENTATION (PIPELINE V2.1)")
    print("=" * 80)
    print(f"Input Directory  : {input_dir.resolve()}")
    print(f"Output Directory : {output_dir.resolve()}")
    print(f"Target Classes   : {len(class_dirs)}")
    if fixed_test:
        n_test_found = sum(len(v) for v in test_hashes.values())
        print(f"Test Directory   : {test_input.resolve()}  ({n_test_found} images, fixed)")
        print(f"Split Ratios     : Train={train_ratio*100:.0f}%, Val={val_ratio*100:.0f}% "
              f"(of clean_data after removing test overlap); Test = supplied folder")
    else:
        print(f"Split Ratios     : Train={train_ratio*100:.0f}%, Val={val_ratio*100:.0f}%, "
              f"Test={test_ratio*100:.0f}%")
    print(f"Augmentation     : {args.num_augments}x (TRAIN ONLY)")
    print(f"Image Resolution : {args.target_size}x{args.target_size} (Aspect-preserving pad)")
    print(f"Random Seed      : {args.seed}")
    print("=" * 80)

    within_test_dupes = sum(len(v) - 1 for v in test_hashes.values())
    if within_test_dupes:
        print(f"[warn] {within_test_dupes} test image(s) are byte-identical to another "
              f"test image; the test set is kept as supplied.")

    output_dir.mkdir(parents=True, exist_ok=True)
    train_dir = output_dir / "train"
    val_dir = output_dir / "val"
    test_dir = output_dir / "test"
    for d in [train_dir, val_dir, test_dir]:
        d.mkdir(parents=True, exist_ok=True)

    manifest = {
        "config": {
            "mode": "train_val_with_fixed_test" if fixed_test else "train_val_test",
            "train_ratio": train_ratio,
            "val_ratio": val_ratio,
            "test_ratio": test_ratio,
            "test_source": str(test_input) if fixed_test else None,
            "num_augments": args.num_augments,
            "target_size": args.target_size,
            "seed": args.seed,
        },
        "classes": {},
        "removed": {"overlap_with_test": [], "duplicates_in_pool": []},
        "totals": {}
    }

    min_images = 2 if fixed_test else 3
    summary_rows = []
    grand_train_orig, grand_train_aug, grand_val, grand_test = 0, 0, 0, 0
    grand_skipped = 0
    removed_test, removed_dupes = [], []

    for c_dir in class_dirs:
        class_name = c_dir.name

        # Build the pool: drop images that also appear in the fixed test set, and
        # collapse byte-identical duplicates so they cannot straddle train/val.
        img_paths, digests, seen = [], {}, set()
        for f in list_images(c_dir):
            digest = file_md5(f)
            if digest in test_hashes:
                removed_test.append(f"{class_name}/{f.name}")
                continue
            if digest in seen:
                removed_dupes.append(f"{class_name}/{f.name}")
                continue
            seen.add(digest)
            img_paths.append(f)
            digests[f] = digest
        n_total = len(img_paths)

        if n_total < min_images:
            print(f"  [skip] Class '{class_name}' has too few usable images ({n_total})")
            continue

        # Create class folders in splits
        c_train = train_dir / class_name
        c_val = val_dir / class_name
        c_train.mkdir(parents=True, exist_ok=True)
        c_val.mkdir(parents=True, exist_ok=True)
        c_test = test_dir / class_name
        if not fixed_test:
            c_test.mkdir(parents=True, exist_ok=True)

        # Fixed test:  one split  -> Train vs Val
        # Three-way:   first split -> Train vs Temp (Val + Test), then Temp -> Val vs Test
        # train_test_split raises when a requested side rounds down to zero, so
        # a class with very few images is reported and skipped rather than
        # killing the run partway through.
        try:
            if fixed_test:
                train_files, val_files = train_test_split(
                    img_paths,
                    train_size=train_ratio,
                    random_state=args.seed,
                    shuffle=True
                )
                test_files = []
            else:
                val_test_ratio = val_ratio + test_ratio
                val_proportion_of_temp = val_ratio / val_test_ratio
                train_files, temp_files = train_test_split(
                    img_paths,
                    train_size=train_ratio,
                    random_state=args.seed,
                    shuffle=True
                )
                val_files, test_files = train_test_split(
                    temp_files,
                    train_size=val_proportion_of_temp,
                    random_state=args.seed,
                    shuffle=True
                )
        except ValueError as exc:
            print(f"  [skip] Class '{class_name}' ({n_total} images) cannot be split "
                  f"with ratios {train_ratio:.0%}/{val_ratio:.0%}/{test_ratio:.0%}: {exc}")
            continue

        used_stems = set()      # keeps output stems unique within this class

        # Process Train: Base + Augmented
        train_orig_count = 0
        train_aug_count = 0
        for f in train_files:
            img_bgr = process_image_file(f, args.target_size)
            if img_bgr is None:
                continue
            stem = unique_stem(f.stem, digests[f], used_stems)

            # Base training image
            write_jpg(c_train / f"{stem}.jpg", img_bgr)
            train_orig_count += 1

            # Augmented variants
            for aug_i in range(1, args.num_augments + 1):
                aug_bgr = augment_image(img_bgr)
                write_jpg(c_train / f"{stem}_aug{aug_i}.jpg", aug_bgr)
                train_aug_count += 1

        # Process Val: Pure Originals Only (No augmentation)
        val_count = 0
        for f in val_files:
            img_bgr = process_image_file(f, args.target_size)
            if img_bgr is None:
                continue
            stem = unique_stem(f.stem, digests[f], used_stems)
            write_jpg(c_val / f"{stem}.jpg", img_bgr)
            val_count += 1

        # Process Test (three-way mode only): Pure Originals Only (No augmentation)
        test_count = 0
        for f in test_files:
            img_bgr = process_image_file(f, args.target_size)
            if img_bgr is None:
                continue
            stem = unique_stem(f.stem, digests[f], used_stems)
            write_jpg(c_test / f"{stem}.jpg", img_bgr)
            test_count += 1

        # Record manifest
        manifest["classes"][class_name] = {
            "total_originals": n_total,
            "train_originals": len(train_files),
            "train_augmented": train_aug_count,
            "train_total": train_orig_count + train_aug_count,
            "val_count": val_count,
            "test_count": test_count,
            "train_files": [f.name for f in train_files],
            "val_files": [f.name for f in val_files],
            "test_files": [f.name for f in test_files],
        }

        grand_train_orig += train_orig_count
        grand_train_aug += train_aug_count
        grand_val += val_count
        grand_test += test_count
        grand_skipped += ((len(train_files) - train_orig_count)
                          + (len(val_files) - val_count)
                          + (len(test_files) - test_count))

        summary_rows.append({
            "class": class_name,
            "originals": n_total,
            "train_orig": train_orig_count,
            "train_aug": train_aug_count,
            "train_total": train_orig_count + train_aug_count,
            "val": val_count,
            "test": test_count,
        })

    # Fixed test set: standardize the supplied images (no augmentation)
    if fixed_test:
        missing = sorted({d.name for d in test_class_dirs} - set(manifest["classes"]))
        if missing:
            die(f"These classes have test images but were skipped for training: {missing}")

        print("\nProcessing fixed test set ...")
        for d in test_class_dirs:
            c_test = test_dir / d.name
            c_test.mkdir(parents=True, exist_ok=True)
            used_stems = set()
            names = []
            for f in list_images(d):
                img_bgr = process_image_file(f, args.target_size)
                if img_bgr is None:
                    grand_skipped += 1
                    continue
                stem = unique_stem(f.stem, file_md5(f), used_stems)
                write_jpg(c_test / f"{stem}.jpg", img_bgr)
                names.append(f.name)
            manifest["classes"][d.name]["test_count"] = len(names)
            manifest["classes"][d.name]["test_files"] = names
            grand_test += len(names)
        for row in summary_rows:
            row["test"] = manifest["classes"][row["class"]]["test_count"]

    if removed_test:
        print(f"\n[info] Removed {len(removed_test)} clean_data image(s) that are "
              f"byte-identical to a test image.")
    if removed_dupes:
        print(f"[info] Removed {len(removed_dupes)} byte-identical duplicate(s) from the "
              f"train/val pool.")
    if grand_skipped:
        print(f"\n[warn] {grand_skipped} image(s) could not be processed and were "
              f"left out of the splits.")

    # Save manifest
    manifest["removed"] = {
        "overlap_with_test": removed_test,
        "duplicates_in_pool": removed_dupes,
    }
    manifest["totals"] = {
        "grand_train_originals": grand_train_orig,
        "grand_train_augmented": grand_train_aug,
        "grand_train_total": grand_train_orig + grand_train_aug,
        "grand_val_total": grand_val,
        "grand_test_total": grand_test,
        "grand_skipped": grand_skipped,
        "grand_removed_test_overlap": len(removed_test),
        "grand_removed_duplicates": len(removed_dupes),
    }
    manifest_path = output_dir / "split_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    # Print Summary Table
    orig_label = "Pool" if fixed_test else "Originals"
    print("\n" + "=" * 80)
    print(f"{'Class Name':<28} {orig_label:>10} {'Train Base':>12} {'Train Aug':>10} {'Train Total':>12} {'Val (Pure)':>11} {'Test (Pure)':>11}")
    print("-" * 80)
    for r in summary_rows:
        print(f"{r['class']:<28} {r['originals']:>10} {r['train_orig']:>12} {r['train_aug']:>10} {r['train_total']:>12} {r['val']:>11} {r['test']:>11}")
    print("-" * 80)
    print(f"{'TOTAL':<28} {sum(r['originals'] for r in summary_rows):>10} {grand_train_orig:>12} {grand_train_aug:>10} {grand_train_orig + grand_train_aug:>12} {grand_val:>11} {grand_test:>11}")
    print("=" * 80)
    if fixed_test:
        print("Pool = train + val originals left in clean_data after removing test overlap and duplicates.")
    print(f"Manifest written to: {manifest_path.resolve()}")
    print("Data splitting & selective augmentation complete!\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/pitha_scripts/pitha_metadata.json
{
  "bhapa_pitha": {
    "display_name": "Bhapa Pitha",
    "alt_names": [],
    "district": "Cumilla",
    "division": "Chittagong",
    "cooking_method": "steamed",
    "key_ingredients": [
      "rice flour",
      "coconut (grated)",
      "date palm jaggery"
    ],
    "recipe_summary": "Damp rice flour is layered in a mold with grated coconut and jaggery filling in the center, then steamed over boiling water for 3-5 minutes. Served hot.",
    "calories": 180,
    "protein_g": 3,
    "carbs_g": 34,
    "fat_g": 3,
    "sugar_g": 12
  },
  "bibikhana_pitha": {
    "display_name": "Bibikhana Pitha",
    "alt_names": [],
    "district": "Jamalpur",
    "division": "Mymensingh",
    "cooking_method": "baked",
    "key_ingredients": [
      "rice flour",
      "coconut (grated)",
      "date palm jaggery",
      "milk",
      "eggs",
      "ghee",
      "cardamom"
    ],
    "recipe_summary": "Rice flour, coconut, jaggery, eggs, and milk are mixed into a thick batter, poured into a greased baking dish, and baked at 180C for 35-45 minutes until golden. Cut into squares.",
    "calories": 260,
    "protein_g": 5,
    "carbs_g": 40,
    "fat_g": 9,
    "sugar_g": 16
  },
  "binni_chaler_pitha": {
    "display_name": "Binni Chaler Pitha",
    "alt_names": [],
    "district": "Barguna",
    "division": "Barishal",
    "cooking_method": "pan_cooked",
    "key_ingredients": [
      "binni chal (sticky rice)",
      "coconut (grated)",
      "date palm jaggery"
    ],
    "recipe_summary": "Coarsely ground sticky rice is dampened, spread thinly on a dry pan, steamed briefly, then filled with grated coconut and jaggery, folded into a half-moon shape.",
    "calories": 200,
    "protein_g": 3,
    "carbs_g": 40,
    "fat_g": 4,
    "sugar_g": 12
  },
  "chanamukhi": {
    "display_name": "Chanamukhi",
    "alt_names": [
      "Chhanamukhi",
      "Chanamukhi Pitha"
    ],
    "district": "Brahmanbaria",
    "division": "Chittagong",
    "cooking_method": "simmered_in_sugar_syrup",
    "key_ingredients": [
      "cow's milk chhana (curd cheese)",
      "sugar",
      "cardamom",
      "flour"
    ],
    "recipe_summary": "Cow's milk is boiled and curdled, and the chhana is hung in cloth until it firms up, then cut into small four-cornered pieces. The pieces are simmered in cardamom-scented sugar syrup for 15-20 minutes and dried in open air, which leaves a hardened sugar crust; about 7-8 litres of milk and 1 kg of sugar yield 1 kg of chanamukhi. A Brahmanbaria speciality since British times, it received Bangladesh's geographical indication (GI) recognition as GI No. 41 on 24 September 2024.",
    "calories": 210,
    "protein_g": 8,
    "carbs_g": 29,
    "fat_g": 7,
    "sugar_g": 25
  },
  "chitoi": {
    "display_name": "Chitoi",
    "alt_names": [
      "Chitoi Pitha"
    ],
    "district": "Chuadanga",
    "division": "Khulna",
    "cooking_method": "pan_cooked",
    "key_ingredients": [
      "rice flour",
      "water",
      "salt"
    ],
    "recipe_summary": "A simple pitha: smooth rice flour batter is poured into a hot clay mold or pan, covered, and cooked until the surface becomes firm and slightly porous. Served with bhorta, curry, or jaggery.",
    "calories": 140,
    "protein_g": 2,
    "carbs_g": 30,
    "fat_g": 1,
    "sugar_g": 0
  },
  "chushi_pitha": {
    "display_name": "Chushi Pitha",
    "alt_names": [],
    "district": "Mymensingh",
    "division": "Mymensingh",
    "cooking_method": "boiled_in_milk",
    "key_ingredients": [
      "rice flour",
      "milk",
      "date palm jaggery",
      "cardamom",
      "coconut"
    ],
    "recipe_summary": "Tiny elongated pieces of rice flour dough are boiled in water, drained, then simmered in sweetened reduced milk with jaggery, cardamom, and optional coconut.",
    "calories": 235,
    "protein_g": 5,
    "carbs_g": 40,
    "fat_g": 6,
    "sugar_g": 17
  },
  "dim_sundori_pitha_pantowa": {
    "display_name": "Dim Sundori Pitha (Pantowa)",
    "alt_names": [
      "Pantowa"
    ],
    "district": "Barishal",
    "division": "Barishal",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "all-purpose flour",
      "egg",
      "milk",
      "sugar",
      "hard-boiled eggs",
      "cardamom"
    ],
    "recipe_summary": "Hard-boiled eggs are repeatedly dipped in a sweet flour-milk batter and deep-fried in cycles (4-6 times), building concentric golden layers. Sliced cross-sectionally to reveal rings.",
    "calories": 280,
    "protein_g": 8,
    "carbs_g": 35,
    "fat_g": 12,
    "sugar_g": 16
  },
  "dudh_chitoi": {
    "display_name": "Dudh Chitoi",
    "alt_names": [],
    "district": "Sylhet",
    "division": "Sylhet",
    "cooking_method": "pan_cooked_then_soaked",
    "key_ingredients": [
      "rice flour",
      "full-fat milk",
      "date palm jaggery",
      "cardamom",
      "coconut"
    ],
    "recipe_summary": "Porous rice flour pithas are cooked on a clay mold, then soaked overnight in reduced milk sweetened with date palm jaggery and cardamom. Served cold.",
    "calories": 250,
    "protein_g": 5,
    "carbs_g": 42,
    "fat_g": 7,
    "sugar_g": 20
  },
  "gulgula": {
    "display_name": "Gulgula",
    "alt_names": [],
    "district": "Lakshmipur",
    "division": "Chittagong",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "all-purpose flour",
      "date palm jaggery",
      "coconut (grated)",
      "fennel seeds"
    ],
    "recipe_summary": "Jaggery water is mixed with rice flour, all-purpose flour, coconut, and fennel into a thick batter, rested, then spoonfuls are dropped into hot oil and fried until deep golden brown.",
    "calories": 210,
    "protein_g": 3,
    "carbs_g": 33,
    "fat_g": 8,
    "sugar_g": 13
  },
  "jhal_jamai_pitha_chapri": {
    "display_name": "Jhal Jamai Pitha (Chapri)",
    "alt_names": [
      "Chapri"
    ],
    "district": "Magura",
    "division": "Khulna",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "onion",
      "green chili",
      "coriander",
      "turmeric",
      "nigella seeds"
    ],
    "recipe_summary": "A savory pitha: spiced rice flour dough with onion, chili, coriander, and turmeric is flattened into thin ovals (chapri) and fried until both sides are golden brown and crisp.",
    "calories": 185,
    "protein_g": 3,
    "carbs_g": 28,
    "fat_g": 7,
    "sugar_g": 1
  },
  "jhinuk_pitha": {
    "display_name": "Jhinuk Pitha",
    "alt_names": [],
    "district": "Khulna",
    "division": "Khulna",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "all-purpose flour",
      "milk",
      "coconut (grated)",
      "sugar",
      "cardamom"
    ],
    "recipe_summary": "Flour-coconut dough is shaped into small ovals pressed with two clean combs to create a shell-like (jhinuk/mussel) pattern, deep-fried until golden, then rolled in warm sugar-cardamom syrup.",
    "calories": 235,
    "protein_g": 4,
    "carbs_g": 36,
    "fat_g": 9,
    "sugar_g": 16
  },
  "khejuri_pitha": {
    "display_name": "Khejuri Pitha",
    "alt_names": [],
    "district": "Barishal",
    "division": "Barishal",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "date palm jaggery",
      "coconut (grated)",
      "water"
    ],
    "recipe_summary": "Rice flour dough is shaped into small ovals or leaf-like forms, deep-fried until golden and crisp, then briefly dipped in warm date palm jaggery syrup.",
    "calories": 215,
    "protein_g": 3,
    "carbs_g": 34,
    "fat_g": 8,
    "sugar_g": 14
  },
  "khola_jali_pitha": {
    "display_name": "Khola Jali Pitha",
    "alt_names": [],
    "district": "Patuakhali",
    "division": "Barishal",
    "cooking_method": "pan_cooked",
    "key_ingredients": [
      "rice flour",
      "onion",
      "green chili",
      "coriander leaves",
      "nigella seeds"
    ],
    "recipe_summary": "A savory pitha: thin rice flour batter with onion, chili, and coriander is poured onto a hot tawa and cooked covered on both sides until lightly crisp. Served with chutney or bhorta.",
    "calories": 160,
    "protein_g": 3,
    "carbs_g": 30,
    "fat_g": 3,
    "sugar_g": 1
  },
  "malpoya": {
    "display_name": "Malpoya",
    "alt_names": [
      "Malpua"
    ],
    "district": "Jhenaidah",
    "division": "Khulna",
    "cooking_method": "pan_fried",
    "key_ingredients": [
      "all-purpose flour",
      "rice flour",
      "semolina",
      "milk",
      "sugar",
      "fennel seeds",
      "cardamom"
    ],
    "recipe_summary": "Thick batter of flour, semolina, milk, sugar, and fennel is rested, then spoonfuls are shallow-fried in ghee/oil until golden on both sides. Served warm, optionally with jaggery syrup.",
    "calories": 230,
    "protein_g": 4,
    "carbs_g": 35,
    "fat_g": 8,
    "sugar_g": 14
  },
  "mera_pitha": {
    "display_name": "Mera Pitha",
    "alt_names": [],
    "district": "Sunamganj",
    "division": "Sylhet",
    "cooking_method": "steamed",
    "key_ingredients": [
      "rice flour",
      "coconut (grated)",
      "jaggery (gur)",
      "water"
    ],
    "recipe_summary": "Rice flour dough is kneaded with grated coconut and jaggery, shaped into elongated ovals with pointed ends, then steamed over boiling water for 12-15 minutes until firm.",
    "calories": 175,
    "protein_g": 3,
    "carbs_g": 35,
    "fat_g": 3,
    "sugar_g": 10
  },
  "muittha_pitha": {
    "display_name": "Muittha Pitha",
    "alt_names": [],
    "district": "Barguna",
    "division": "Barishal",
    "cooking_method": "steamed",
    "key_ingredients": [
      "rice flour",
      "coconut (grated)",
      "date palm jaggery"
    ],
    "recipe_summary": "Rice flour is cooked with coconut and jaggery into a dough, shaped by squeezing in the palm to create finger-impression patterns, then steamed for 12-15 minutes.",
    "calories": 185,
    "protein_g": 3,
    "carbs_g": 36,
    "fat_g": 3,
    "sugar_g": 13
  },
  "narkel_puli_pitha": {
    "display_name": "Narkel Puli Pitha",
    "alt_names": [
      "Narkel Puli"
    ],
    "district": "Sunamganj",
    "division": "Sylhet",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "coconut (grated)",
      "date palm jaggery (khejur gur)",
      "cardamom"
    ],
    "recipe_summary": "Coconut-jaggery filling is wrapped in rice flour dough shaped into crescent/half-moon shapes with patterned edges, then deep-fried until golden brown and crispy.",
    "calories": 220,
    "protein_g": 3,
    "carbs_g": 32,
    "fat_g": 9,
    "sugar_g": 14
  },
  "nunia_pitha": {
    "display_name": "Nunia Pitha",
    "alt_names": [],
    "district": "Sylhet",
    "division": "Sylhet",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "onion paste",
      "ginger paste",
      "garlic paste",
      "turmeric",
      "green chilies",
      "nigella seeds"
    ],
    "recipe_summary": "A savory pitha: spiced rice flour dough with onion, ginger, garlic, and coriander is rolled thin, cut into diamond or flower shapes, and deep-fried until golden-yellow and puffy.",
    "calories": 190,
    "protein_g": 3,
    "carbs_g": 28,
    "fat_g": 7,
    "sugar_g": 1
  },
  "patishapta": {
    "display_name": "Patishapta",
    "alt_names": [],
    "district": "Sylhet",
    "division": "Sylhet",
    "cooking_method": "pan_cooked",
    "key_ingredients": [
      "rice flour",
      "all-purpose flour",
      "semolina",
      "milk",
      "date palm jaggery",
      "cardamom"
    ],
    "recipe_summary": "Thin crepe-like wrappers made from rice flour, semolina, and milk are filled with thickened kheer (reduced milk with jaggery and cardamom), then rolled into neat cylinders.",
    "calories": 210,
    "protein_g": 4,
    "carbs_g": 38,
    "fat_g": 5,
    "sugar_g": 15
  },
  "rosgoja_pitha": {
    "display_name": "Rosgoja Pitha",
    "alt_names": [],
    "district": "Jhalokathi",
    "division": "Barishal",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "coconut (grated)",
      "date palm jaggery",
      "cardamom"
    ],
    "recipe_summary": "Rice flour dough balls are deep-fried until golden and crisp, then soaked in warm jaggery-cardamom syrup for 10-15 minutes so the syrup penetrates the crispy exterior.",
    "calories": 225,
    "protein_g": 3,
    "carbs_g": 36,
    "fat_g": 8,
    "sugar_g": 15
  },
  "sorbhaja": {
    "display_name": "Sorbhaja",
    "alt_names": [],
    "district": "Rangpur",
    "division": "Rangpur",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "all-purpose flour",
      "milk",
      "sugar",
      "cardamom",
      "ghee"
    ],
    "recipe_summary": "Rice flour and all-purpose flour dough with milk, sugar, and cardamom is rolled thin, cut into small rectangles or traditional shapes, and deep-fried slowly until crisp and golden.",
    "calories": 220,
    "protein_g": 3,
    "carbs_g": 33,
    "fat_g": 9,
    "sugar_g": 12
  },
  "sotin_mochor": {
    "display_name": "Sotin Mochor",
    "alt_names": [],
    "district": "Bogura",
    "division": "Rajshahi",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "coconut (grated)",
      "date palm jaggery"
    ],
    "recipe_summary": "Rice flour dough is filled with coconut-jaggery filling, folded into layered or twisted shapes with firmly sealed edges, then deep-fried slowly until golden and crisp.",
    "calories": 210,
    "protein_g": 3,
    "carbs_g": 34,
    "fat_g": 8,
    "sugar_g": 13
  },
  "syringe_pitha": {
    "display_name": "Syringe Pitha",
    "alt_names": [],
    "district": "Mymensingh",
    "division": "Mymensingh",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "rice flour",
      "coconut (grated)",
      "date palm jaggery"
    ],
    "recipe_summary": "Cooked rice flour dough is piped through a syringe or piping bag into coils, spirals, or lattice shapes directly into hot oil, fried until crisp, then dipped in jaggery syrup.",
    "calories": 215,
    "protein_g": 3,
    "carbs_g": 34,
    "fat_g": 8,
    "sugar_g": 13
  },
  "taler_cone_pitha": {
    "display_name": "Taler Cone Pitha",
    "alt_names": [],
    "district": "Bhola",
    "division": "Barishal",
    "cooking_method": "deep_fried",
    "key_ingredients": [
      "ripe palm fruit pulp (tal)",
      "rice flour",
      "all-purpose flour",
      "sugar",
      "coconut"
    ],
    "recipe_summary": "Ripe palm fruit pulp is mixed with rice flour, coconut, and sugar into a dough, shaped into pointed cones, and deep-fried until deep golden brown.",
    "calories": 230,
    "protein_g": 3,
    "carbs_g": 38,
    "fat_g": 8,
    "sugar_g": 14
  }
}

## 1. Environment Setup

This section imports the required libraries, verifies that the installed TensorFlow build provides the Xception architecture and its preprocessing function, and enables incremental GPU memory allocation when a GPU is available. Feature extraction with the frozen backbone also runs on a CPU, only more slowly.

In [ ]:
import sys, os, json, math, time, random, platform, subprocess, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import PIL
from IPython.display import display

# scikit-learn >= 1.9 emits a FutureWarning for SVC(probability=True). The option is still
# supported and is required here to obtain class-probability estimates (ROC-AUC, confidence).
warnings.filterwarnings('ignore', message='.*probability.*', category=FutureWarning)

import tensorflow as tf

# Enable incremental GPU memory allocation (if a GPU is present) instead of reserving all memory at start-up
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass  # must be set before the GPU is initialized; safe to ignore if already done

# XGBoost trains on the GPU when one is available (much faster than Colab's 2-core CPU)
XGB_DEVICE = 'cuda' if gpus else 'cpu'
# XGBoost warns when a GPU model predicts from CPU arrays; the result is unaffected
warnings.filterwarnings('ignore', message='.*mismatched devices.*')

# Verify that the required Xception components exist in this TensorFlow build
if not hasattr(tf.keras.applications, 'Xception'):
    raise RuntimeError(f'This TensorFlow build ({tf.__version__}) does not provide Xception.')
if not hasattr(tf.keras.applications.xception, 'preprocess_input'):
    raise RuntimeError('keras.applications.xception.preprocess_input is not available.')

import sklearn
import joblib
import xgboost
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, top_k_accuracy_score)

print(f'Python      : {sys.version.split()[0]}')
print(f'TensorFlow  : {tf.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'XGBoost     : {xgboost.__version__}')
print(f'Device      : {"GPU" if gpus else "CPU (no GPU detected)"}')
print(f'XGBoost on  : {XGB_DEVICE}')
if not gpus:
    print('Tip: select a GPU runtime (Runtime > Change runtime type > T4 GPU) - it is much faster.')
print('Xception and its preprocessing function are available.')

## 2. Configuration

The input folders on Google Drive were located in Section 0. Everything this notebook generates is written to the local Colab disk under `/content/`, which is much faster than Drive, and is packed into a zip at the end. The cell below also creates the output folders:

- `model_xception/` – trained models and intermediate artifacts
- `model_xception/figures/` – figures (PNG), saved automatically whenever they are displayed
- `model_xception/tables/` – result tables (CSV), saved automatically whenever they are displayed

All experiment settings are collected in this single cell so that every run is fully specified by it.

In [ ]:
# ---------------------------------------------------------------- paths (Google Colab)
# Inputs are read from Google Drive (located in Section 0). Everything generated is written to the
# local Colab disk, which is much faster than Drive, and packed into a zip at the end of the notebook.
BASE_DIR   = Path('/content')
SCRIPT_DIR = BASE_DIR / 'pitha_scripts'     # pitha_preprocessing.py, split_and_augment.py, pitha_metadata.json

CURATED_DIR    = CLEAN_DATA_DIR             # Drive: clean_data/<class>/ - training + validation pool
TEST_INPUT_DIR = TEST_DATA_DIR              # Drive: split_data_v2/test/<class>/ - test set selected by hand
SPLIT_DIR   = BASE_DIR / 'split_data'       # generated train / val / test folders
MODEL_DIR   = BASE_DIR / 'outputs' / 'model_xception'   # trained models and intermediate artifacts
FIG_DIR     = MODEL_DIR / 'figures'         # report figures (PNG)
TABLE_DIR   = MODEL_DIR / 'tables'          # report tables (CSV)

for folder in (MODEL_DIR, FIG_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- experiment settings
TRAIN_RATIO, VAL_RATIO = 0.89, 0.11   # split of the curated pool; the test set is fixed (selected by hand)
NUM_AUGMENTS   = 5      # augmented copies generated per training image
TARGET_SIZE    = 224    # backbone input resolution (pixels)
PCA_COMPONENTS = 256    # number of principal components retained
BATCH_SIZE     = 16     # images per batch during feature extraction
MISCLASSIFIED_PER_PAGE = 16   # images per figure in the misclassified-image gallery
FIG_DPI        = 300    # resolution of saved figures
SEED           = 42     # random seed used throughout

# ---------------------------------------------------------------- reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------------------------------------------------------- display and plot style
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'savefig.dpi': FIG_DPI, 'savefig.bbox': 'tight', 'figure.dpi': 100})

# ---------------------------------------------------------------- sanity checks
for required_dir in (CURATED_DIR, TEST_INPUT_DIR):
    if not required_dir.exists():
        raise FileNotFoundError(f'Dataset folder not found: {required_dir}. Run the cells of Section 0 first.')
for required_file in ('pitha_preprocessing.py', 'split_and_augment.py', 'pitha_metadata.json'):
    if not (SCRIPT_DIR / required_file).exists():
        raise FileNotFoundError(f'{required_file} not found in {SCRIPT_DIR}. Run the cells of Section 0 first.')

# make pitha_preprocessing.py importable
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

class_folders = sorted(d.name for d in CURATED_DIR.iterdir() if d.is_dir())
print(f'Train/val pool: {CURATED_DIR}')
print(f'Test set      : {TEST_INPUT_DIR}')
print(f'Outputs       : {MODEL_DIR}')
print(f'Classes found: {len(class_folders)}')

The experimental setup and the software environment are recorded as tables so that they can be reported directly.

In [ ]:
# ---------------------------------------------------------------- experimental setup table
setup_table = pd.DataFrame([
    ('Backbone',            'Xception (ImageNet weights, frozen)',       'Fixed feature extractor, classification head removed'),
    ('Feature dimension',   '2048',                                       'Global average pooling output'),
    ('Input size',          f'{TARGET_SIZE} x {TARGET_SIZE} px',          'Images are resized with padding'),
    ('Dimensionality reduction', f'PCA, {PCA_COMPONENTS} components',      'Fitted on training features only'),
    ('Test set',            'Selected by hand (fixed folder)',           'Never augmented; identical copies removed from the train/val pool'),
    ('Data split (train/val)', f'{TRAIN_RATIO:.2f} / {VAL_RATIO:.2f}',   'Of the curated pool, stratified by class'),
    ('Augmentation',        f'{NUM_AUGMENTS} copies per training image',  'Training subset only, applied after the split'),
    ('Batch size',          str(BATCH_SIZE),                              'Feature extraction'),
    ('Random seed',         str(SEED),                                    'Split, PCA, classifiers'),
], columns=['parameter', 'value', 'note'])

display(setup_table)
setup_table.to_csv(TABLE_DIR / 'experimental_setup.csv', index=False)

# ---------------------------------------------------------------- software environment table
environment_table = pd.DataFrame([
    ('Python',       sys.version.split()[0]),
    ('Platform',     platform.platform()),
    ('TensorFlow',   tf.__version__),
    ('NumPy',        np.__version__),
    ('pandas',       pd.__version__),
    ('scikit-learn', sklearn.__version__),
    ('XGBoost',      xgboost.__version__),
    ('Matplotlib',   matplotlib.__version__),
    ('seaborn',      sns.__version__),
    ('Pillow',       PIL.__version__),
    ('joblib',       joblib.__version__),
    ('Compute device', 'GPU' if gpus else 'CPU'),
    ('GPU model',    tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'GPU') if gpus else '-'),
    ('XGBoost device', XGB_DEVICE),
    ('Runtime',      'Google Colab' if IN_COLAB else 'local'),
], columns=['component', 'version'])

display(environment_table)
environment_table.to_csv(TABLE_DIR / 'software_environment.csv', index=False)

## 3. Data Preparation: Hand-Selected Test Set, Stratified Split and Augmentation

The **test set was selected by hand** and is read unchanged from `split_data_v2/test/` on Google Drive. It is passed to `split_and_augment.py` through `--test-input`, so the script uses it exactly as supplied: test images are only standardized (EXIF orientation, 224 × 224 reflection padding), never augmented, and every image in `clean_data/` that is byte-identical to a test image is removed from the training/validation pool, so the test set cannot leak into training.

The curated pool (`clean_data/`) is split by class into training and validation subsets (89 / 11) and the split is seeded for reproducibility. Augmentation is applied **after** the split and **only** to the training images, so the validation and test subsets consist exclusively of original, unmodified images.

The split is written to the local Colab disk (`/content/split_data/`). Every step is seeded, so running this cell in another notebook or session reproduces exactly the same partition; the **split fingerprint** printed below lets you confirm that all six backbone notebooks used an identical split. If `split_manifest.json` already exists in the current session, the step is skipped. The script's console output is displayed below.

In [ ]:
manifest_path = SPLIT_DIR / 'split_manifest.json'

if manifest_path.exists():
    print(f'Split already present at {SPLIT_DIR} - skipping.')
    print('Delete the split_data folder to rebuild it from scratch.')
else:
    # Build the command as a list of separate arguments (avoids quoting problems with spaces in paths).
    # sys.executable is the Python interpreter running this notebook, so the same packages are available.
    cmd = [
        sys.executable, str(SCRIPT_DIR / 'split_and_augment.py'),
        '--input', str(CURATED_DIR),
        '--test-input', str(TEST_INPUT_DIR),     # test set selected by hand, used as supplied
        '--output', str(SPLIT_DIR),
        '--train-ratio', str(TRAIN_RATIO),
        '--val-ratio', str(VAL_RATIO),
        '--num-augments', str(NUM_AUGMENTS),
        '--target-size', str(TARGET_SIZE),
        '--seed', str(SEED),
    ]
    print('Running:', ' '.join(cmd), '\n')

    # Run the script and capture its output so that it appears in this cell.
    # UTF-8 is forced on both sides so that non-ASCII characters printed by the script cannot cause
    # encoding errors (a common problem with the default console encoding on Windows).
    child_env = os.environ.copy()
    child_env['PYTHONIOENCODING'] = 'utf-8'
    result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace', env=child_env)
    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f'split_and_augment.py exited with code {result.returncode} - see the output above.')

The split manifest written by the script is read back to verify the subset sizes. The summary is saved as a table.

In [ ]:
with open(manifest_path) as f:
    manifest = json.load(f)

totals = manifest['totals']
train_orig  = totals['grand_train_originals']
train_aug   = totals['grand_train_augmented']
train_total = totals['grand_train_total']
val_total   = totals['grand_val_total']
test_total  = totals['grand_test_total']
n_original  = train_orig + val_total + test_total   # number of distinct source images

assert train_total > 0 and val_total > 0 and test_total > 0, 'One of the subsets is empty.'
print(f'Classes: {len(manifest["classes"])}')
if totals.get('grand_skipped'):
    print(f'Unreadable images skipped: {totals["grand_skipped"]}')

split_summary = pd.DataFrame({
    'subset': ['Training - original', 'Training - augmented', 'Training - total',
               'Validation', 'Test', 'All original images'],
    'images': [train_orig, train_aug, train_total, val_total, test_total, n_original],
    'share_of_original_images_pct': [train_orig / n_original * 100, np.nan, np.nan,
                                     val_total / n_original * 100, test_total / n_original * 100, 100.0],
})

display(split_summary.round(2).fillna('-'))     # '-' marks rows where a percentage does not apply
split_summary.to_csv(TABLE_DIR / 'split_summary.csv', index=False)

In [ ]:
# ---------------------------------------------------------------- collected images, cleaning and split fingerprint
import hashlib, shutil

# images per class exactly as collected on Google Drive
raw_pool, raw_test = images_per_class(CURATED_DIR), images_per_class(TEST_INPUT_DIR)
raw_counts = pd.DataFrame({'clean_data_pool': pd.Series(raw_pool),
                           'test_selected_by_hand': pd.Series(raw_test)}).fillna(0).astype(int)
raw_counts['total_collected'] = raw_counts.sum(axis=1)
raw_counts.loc['TOTAL'] = raw_counts.sum()
raw_counts.index.name = 'class'
print('Images per class as collected:')
display(raw_counts)
raw_counts.to_csv(TABLE_DIR / 'raw_input_counts.csv')

# every image that was collected but not used, and why
cleaning = pd.DataFrame([
    ('Images in clean_data/ (training + validation pool)', int(sum(raw_pool.values()))),
    ('Images in the hand-selected test folder',            int(sum(raw_test.values()))),
    ('Total collected images',                             int(sum(raw_pool.values()) + sum(raw_test.values()))),
    ('Pool images identical to a test image (removed)',    int(totals.get('grand_removed_test_overlap', 0))),
    ('Exact duplicates inside the pool (removed)',         int(totals.get('grand_removed_duplicates', 0))),
    ('Unreadable images (skipped)',                        int(totals.get('grand_skipped', 0))),
    ('Original images used - training',                    int(train_orig)),
    ('Original images used - validation',                  int(val_total)),
    ('Original images used - test',                        int(test_total)),
], columns=['item', 'images'])
print('Data cleaning summary:')
display(cleaning)
cleaning.to_csv(TABLE_DIR / 'data_cleaning_summary.csv', index=False)

# fingerprint of the exact file-to-partition assignment: equal in every notebook = identical split
assignment = {c: {k: sorted(v[k]) for k in ('train_files', 'val_files', 'test_files')}
              for c, v in sorted(manifest['classes'].items())}
SPLIT_FINGERPRINT = hashlib.sha256(json.dumps(assignment, sort_keys=True).encode()).hexdigest()[:16]
pd.DataFrame([{'split_fingerprint': SPLIT_FINGERPRINT, 'train_original': int(train_orig),
               'validation': int(val_total), 'test': int(test_total)}]).to_csv(
    TABLE_DIR / 'split_fingerprint.csv', index=False)
print(f'Split fingerprint: {SPLIT_FINGERPRINT}  (the same value in all six notebooks = identical split)')

# keep the split manifest with the results, so the exact partition can always be reconstructed
shutil.copy2(manifest_path, MODEL_DIR / 'split_manifest.json')

## 4. Dataset Inventory and Visualization

This section lists the images of every subset, tabulates and plots the class distribution, and shows example images. Image filenames are numeric serial numbers that are unique only **within** a class folder, so every image is identified by the pair *(class, filename)* throughout the notebook.

Augmented training images are recognized by the `_aug` marker in their filename; all other images are originals.

In [ ]:
from pitha_preprocessing import load_and_orient_image, resize_with_pad

VALID_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def collect_split(split_dir):
    # Returns (paths, labels, filenames) for one split folder; the label is the class folder name
    paths, labels, filenames = [], [], []
    class_dirs = sorted(d for d in split_dir.iterdir() if d.is_dir())
    for c_dir in class_dirs:
        for f in sorted(c_dir.iterdir()):
            if f.suffix.lower() in VALID_EXT:
                paths.append(f)
                labels.append(c_dir.name)
                filenames.append(f.name)
    return paths, labels, filenames

def sample_image_for_class(class_name, split='train'):
    # Returns the path of the first original (non-augmented) image of a class, or None
    class_dir = SPLIT_DIR / split / class_name
    if not class_dir.exists():
        return None
    candidates = sorted(f for f in class_dir.iterdir()
                        if f.suffix.lower() in VALID_EXT and '_aug' not in f.stem)
    return candidates[0] if candidates else None

train_paths, train_labels, train_files = collect_split(SPLIT_DIR / 'train')
val_paths,   val_labels,   val_files   = collect_split(SPLIT_DIR / 'val')
test_paths,  test_labels,  test_files  = collect_split(SPLIT_DIR / 'test')

CLASS_NAMES = sorted(set(train_labels) | set(val_labels) | set(test_labels))

# Flag which training images are originals (True) and which are augmented copies (False)
train_is_original = []
for fname in train_files:
    train_is_original.append('_aug' not in Path(fname).stem)
train_is_original = np.array(train_is_original)

print(f'Classes                        : {len(CLASS_NAMES)}')
print(f'Training images (incl. augmented): {len(train_paths)}  '
      f'({train_is_original.sum()} original + {(~train_is_original).sum()} augmented)')
print(f'Validation images              : {len(val_paths)}')
print(f'Test images                    : {len(test_paths)}')

In [ ]:
# ---------------------------------------------------------------- per-class image counts
train_labels_arr = np.array(train_labels)
val_labels_arr   = np.array(val_labels)
test_labels_arr  = np.array(test_labels)

inventory_rows = []
for cname in CLASS_NAMES:
    n_train_orig = int(((train_labels_arr == cname) & train_is_original).sum())
    n_train_aug  = int(((train_labels_arr == cname) & ~train_is_original).sum())
    n_val        = int((val_labels_arr == cname).sum())
    n_test       = int((test_labels_arr == cname).sum())
    inventory_rows.append([cname, n_train_orig, n_train_aug, n_val, n_test,
                           n_train_orig + n_val + n_test])

inventory = pd.DataFrame(inventory_rows, columns=['class', 'train_original', 'train_augmented',
                                                   'validation', 'test', 'total_original'])

# Append a totals row for the saved / displayed table
total_row = pd.DataFrame([['TOTAL'] + inventory.drop(columns='class').sum().tolist()],
                         columns=inventory.columns)
inventory_with_total = pd.concat([inventory, total_row], ignore_index=True)

display(inventory_with_total)
inventory_with_total.to_csv(TABLE_DIR / 'dataset_summary_by_class.csv', index=False)

# ---------------------------------------------------------------- class distribution chart (original images)
ax = inventory.set_index('class')[['train_original', 'validation', 'test']].plot(
    kind='barh', stacked=True, figsize=(10, 9), color=['#4C72B0', '#DD8452', '#55A868'])
ax.invert_yaxis()                      # keep alphabetical order from top to bottom
ax.set_xlabel('Number of original images')
ax.set_ylabel('')
ax.set_title('Class Distribution of Original Images by Subset')
ax.grid(axis='y', visible=False)
ax.legend(['Training', 'Validation', 'Test'], loc='upper left', bbox_to_anchor=(1.01, 1.0))   # legend outside the bars
plt.tight_layout()
plt.savefig(FIG_DIR / 'class_distribution.png')
plt.show()

In [ ]:
# ---------------------------------------------------------------- one example image per class
n_cols = 6
n_rows = math.ceil(len(CLASS_NAMES) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.3 * n_rows))
axes = axes.reshape(-1)

for i, cname in enumerate(CLASS_NAMES):
    example_path = sample_image_for_class(cname)
    if example_path is not None:
        axes[i].imshow(load_and_orient_image(example_path))
    axes[i].set_title(cname.replace('_', ' '), fontsize=9)
    axes[i].axis('off')

# hide unused axes in the last row
for i in range(len(CLASS_NAMES), len(axes)):
    axes[i].axis('off')

plt.suptitle('Example Image of Each Pitha Class', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'sample_images_per_class.png')
plt.show()

In [ ]:
# ---------------------------------------------------------------- augmentation examples
# Three randomly chosen training images, each shown with its augmented copies
n_examples = 3
rng = np.random.RandomState(SEED)
original_positions = np.where(train_is_original)[0]
chosen_positions = rng.choice(original_positions, size=n_examples, replace=False)

fig, axes = plt.subplots(n_examples, NUM_AUGMENTS + 1,
                         figsize=(3 * (NUM_AUGMENTS + 1), 3.8 * n_examples))

for row, pos in enumerate(chosen_positions):
    original_path = train_paths[pos]
    # augmented copies are stored next to the original, named <original name>_aug*
    augmented_paths = sorted(original_path.parent.glob(original_path.stem + '_aug*'))
    shown_paths = [original_path] + augmented_paths[:NUM_AUGMENTS]

    for col in range(NUM_AUGMENTS + 1):
        axes[row, col].axis('off')
        if col < len(shown_paths):
            axes[row, col].imshow(load_and_orient_image(shown_paths[col]))
            if col == 0:
                axes[row, col].set_title(f'Original\n{train_labels[pos].replace("_", " ")}', fontsize=9)
            else:
                axes[row, col].set_title(f'Augmented {col}', fontsize=9)

plt.suptitle('Training-Set Augmentation Examples', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'augmentation_examples.png')
plt.show()

## 5. Feature Extraction with a Frozen Xception Backbone

Xception is loaded with ImageNet weights and without its classification head. Global average pooling converts the final convolutional feature maps into a **2048-dimensional** vector per image. The backbone is frozen: no fine-tuning is performed, and the network is used purely for inference.

Each image is loaded with EXIF orientation correction, resized with padding to `TARGET_SIZE` × `TARGET_SIZE` pixels (preserving the aspect ratio) and scaled to the range [−1, 1] with `xception.preprocess_input`. Features are extracted for the training set (including augmented images), the validation set and the test set.

In [ ]:
def build_xception_extractor():
    # Frozen Xception without the classification head; global average pooling gives a 2048-d vector
    backbone = tf.keras.applications.Xception(
        include_top=False, weights='imagenet', input_shape=(TARGET_SIZE, TARGET_SIZE, 3),
        pooling='avg')
    backbone.trainable = False
    return backbone

_preprocess = tf.keras.applications.xception.preprocess_input

def extract_features(extractor, paths, batch_size=BATCH_SIZE):
    # Returns an (n_images, 2048) feature matrix for the given list of image paths
    all_feats = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i + batch_size]
        imgs = []
        for p in batch:
            pil_img = load_and_orient_image(p)
            padded = resize_with_pad(np.array(pil_img), size=TARGET_SIZE)   # RGB array
            imgs.append(padded.astype(np.float32))
        arr = np.stack(imgs, axis=0)
        arr = _preprocess(arr)                                    # scale to [-1, 1]
        feats = np.asarray(extractor(arr, training=False))        # inference only
        all_feats.append(feats)
        if (i // batch_size) % 20 == 0 and i > 0:
            print(f'    {min(i + batch_size, len(paths))}/{len(paths)} images processed...')
    return np.vstack(all_feats)

print('Building frozen Xception backbone...')
extractor = build_xception_extractor()
print(f'Pooled feature dimension: {extractor.output_shape[-1]}')

In [ ]:
start_time = time.time()

print('Extracting TRAIN features...')
X_train_raw = extract_features(extractor, train_paths)
print('Extracting VALIDATION features...')
X_val_raw = extract_features(extractor, val_paths)
print('Extracting TEST features...')
X_test_raw = extract_features(extractor, test_paths)

extraction_minutes = (time.time() - start_time) / 60
print(f'\nFeature extraction finished in {extraction_minutes:.1f} minutes.')

feature_shapes = pd.DataFrame({
    'subset': ['Training (incl. augmented)', 'Validation', 'Test'],
    'images': [X_train_raw.shape[0], X_val_raw.shape[0], X_test_raw.shape[0]],
    'feature_dimension': [X_train_raw.shape[1], X_val_raw.shape[1], X_test_raw.shape[1]],
})
display(feature_shapes)
feature_shapes.to_csv(TABLE_DIR / 'feature_matrix_shapes.csv', index=False)

## 6. Dimensionality Reduction with PCA

The 2048-dimensional features are projected onto `PCA_COMPONENTS` principal components. PCA is fitted **exclusively on the training features**; the validation and test features are only transformed. Fitting on evaluation data would let their variance shape the representation and bias the results. The cumulative explained-variance curve and a summary table are saved for reporting.

In [ ]:
# PCA is fitted on the training features only; validation and test features are only transformed
n_components = min(PCA_COMPONENTS, X_train_raw.shape[0], X_train_raw.shape[1])
pca = PCA(n_components=n_components, random_state=SEED)
X_train = pca.fit_transform(X_train_raw)
X_val = pca.transform(X_val_raw)
X_test = pca.transform(X_test_raw)

variance_retained = float(pca.explained_variance_ratio_.astype(np.float64).sum() * 100)
print(f'PCA components   : {n_components}')
print(f'Variance retained: {variance_retained:.2f}%')

# ---------------------------------------------------------------- explained-variance table
variance_ratio = pca.explained_variance_ratio_.astype(np.float64)   # float64 avoids float32 rounding artifacts
variance_table = pd.DataFrame({
    'component': np.arange(1, n_components + 1),
    'explained_variance_pct': variance_ratio * 100,
    'cumulative_variance_pct': np.cumsum(variance_ratio) * 100,
})
variance_table.to_csv(TABLE_DIR / 'pca_explained_variance.csv', index=False)   # full table (all components)

# display only selected milestones to keep the notebook readable
milestone_rows = []
for m in (10, 25, 50, 100, 150, 200, n_components):
    if m <= n_components and (m - 1) not in milestone_rows:
        milestone_rows.append(m - 1)
display(variance_table.iloc[milestone_rows].round(2))

# ---------------------------------------------------------------- persist the transform and the features
joblib.dump(pca, MODEL_DIR / 'pitha_pca_xception.pkl')
np.savez_compressed(MODEL_DIR / 'pitha_features_xception.npz',
                    X_train=X_train, X_val=X_val, X_test=X_test,
                    train_labels=train_labels, val_labels=val_labels, test_labels=test_labels,
                    train_files=train_files, val_files=val_files, test_files=test_files)

backbone_info = {'backbone': 'Xception', 'raw_dims': int(X_train_raw.shape[1]),
                 'pca_components': int(n_components),
                 'variance_retained_pct': float(variance_retained),
                 'preprocess': 'xception.preprocess_input (tf: scale to [-1, 1])'}
with open(MODEL_DIR / 'pitha_backbone_xception.json', 'w') as f:
    json.dump(backbone_info, f, indent=2)

print('\nSaved: pitha_pca_xception.pkl, pitha_features_xception.npz, pitha_backbone_xception.json')

## 7. Classifier Training and Model Selection

Four classifiers are trained on the PCA-reduced training features (original and augmented images). Each classifier is trained once on the full training set and scored on both the validation and the test set. **The classifier with the highest validation accuracy is selected**; test metrics are reported alongside for completeness but play no role in the selection.

Class labels are encoded as integers with a `LabelEncoder`, which is saved together with the selected model.

In [ ]:
# Encode class names as integers (the encoder is saved and reused for inference)
le = LabelEncoder()
y_train = le.fit_transform(train_labels)
y_val = le.transform(val_labels)
y_test = le.transform(test_labels)
joblib.dump(le, MODEL_DIR / 'pitha_label_encoder.pkl')

CLASSIFIERS = {
    'SVM (RBF kernel)': SVC(kernel='rbf', gamma=0.01, C=1.0, probability=True, random_state=SEED),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=SEED),
    'Random Forest (500 trees)': RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED),
    'XGBoost (500 trees)': xgboost.XGBClassifier(
        n_estimators=500, objective='multi:softprob', eval_metric='mlogloss',
        tree_method='hist', device=XGB_DEVICE, n_jobs=-1, random_state=SEED),
}

results = {}
fitted_models = {}

for name, clf in CLASSIFIERS.items():
    print(f'Training: {name} ...')
    clf.fit(X_train, y_train)
    fitted_models[name] = clf

    # predictions on the validation and test sets
    val_pred = clf.predict(X_val)
    test_pred = clf.predict(X_test)

    # accuracy on both sets, plus the macro F1-score on the test set
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)
    _, _, f1_m, _ = precision_recall_fscore_support(y_test, test_pred, average='macro', zero_division=0)

    results[name] = {'val_acc': val_acc, 'test_acc': test_acc, 'test_f1_macro': f1_m}
    print(f'  Validation accuracy: {val_acc*100:.2f}%')
    print(f'  Test accuracy      : {test_acc*100:.2f}%  (macro F1 {f1_m:.4f})\n')

# Comparison table, sorted by validation accuracy (the selection criterion)
results_df = pd.DataFrame(results).T.sort_values('val_acc', ascending=False)
print('Classifier comparison (sorted by validation accuracy - the selection criterion):')
display(results_df.round(4))
results_df.to_csv(TABLE_DIR / 'classifier_comparison.csv')

In [ ]:
# ---------------------------------------------------------------- validation vs. test accuracy chart
accuracy_plot = results_df[['val_acc', 'test_acc']] * 100
accuracy_plot.columns = ['Validation accuracy', 'Test accuracy']

ax = accuracy_plot.plot(kind='bar', figsize=(10, 5.5), width=0.75, color=['#4C72B0', '#DD8452'])
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=9, padding=2)   # value labels on the bars
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.set_xlabel('')
ax.set_title('Classifier Comparison on Xception Features')
plt.xticks(rotation=15, ha='right')
ax.grid(axis='x', visible=False)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))     # legend outside the bars
plt.tight_layout()
plt.savefig(FIG_DIR / 'classifier_accuracy_comparison.png')
plt.show()

In [ ]:
# ---------------------------------------------------------------- model selection (validation accuracy only)
winner_name = results_df.index[0]
winner = fitted_models[winner_name]
print(f'Selected model: {winner_name}')
print(f'  Validation accuracy (selection criterion): {results[winner_name]["val_acc"]*100:.2f}%')
print(f'  Test accuracy (generalization estimate)  : {results[winner_name]["test_acc"]*100:.2f}%')

joblib.dump(winner, MODEL_DIR / 'pitha_model_xception.pkl')
with open(MODEL_DIR / 'pitha_model_xception.info.json', 'w') as f:
    json.dump({'classifier_name': winner_name,
               'val_accuracy': results[winner_name]['val_acc'] * 100,
               'test_accuracy': results[winner_name]['test_acc'] * 100,
               'test_f1_macro': results[winner_name]['test_f1_macro']}, f, indent=2)

print('\nSaved: pitha_model_xception.pkl, pitha_model_xception.info.json')

In [ ]:
import hashlib
def md5(p): return hashlib.md5(p.read_bytes()).hexdigest()

orig_hashes = {}
for split, paths, files in (('train', train_paths, train_files),
                            ('val', val_paths, val_files),
                            ('test', test_paths, test_files)):
    for p, f in zip(paths, files):
        if '_aug' in Path(f).stem:
            continue                      # augmented copies are new pixels, skip them
        orig_hashes.setdefault(md5(p), []).append((split, p.parent.name, f))

dupes = {h: v for h, v in orig_hashes.items() if len({s for s, _, _ in v}) > 1}
print('Identical originals across splits:', len(dupes))
for v in list(dupes.values())[:10]:
    print(v)

## 8. Test-Set Evaluation of the Selected Model

The selected classifier is evaluated on the held-out test set. The following results are reported:

- overall accuracy and top-3 accuracy;
- macro-averaged precision, recall and F1-score;
- per-class precision, recall, F1-score and support;
- the confusion matrix (raw counts);
- one-vs-rest ROC curves and ROC-AUC (macro-average and per class).

Some classes contain only a few test images, so their per-class figures, and in particular the per-class ROC-AUC, carry wide sampling uncertainty and should be interpreted with caution.

In [ ]:
y_test_pred = winner.predict(X_test)
if hasattr(winner, 'predict_proba'):
    y_test_proba = winner.predict_proba(X_test)
else:
    y_test_proba = None
n_classes = len(le.classes_)

# ---------------------------------------------------------------- per-class metrics
report_dict = classification_report(y_test, y_test_pred, target_names=le.classes_,
                                    output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).T
report_df.loc['accuracy', 'support'] = len(y_test)      # the support of the accuracy row is the test-set size

print('Per-class precision, recall and F1-score:')
display(report_df.round(4))
report_df.to_csv(TABLE_DIR / 'per_class_metrics.csv')

# ---------------------------------------------------------------- overall metrics
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_test, y_test_pred, average='macro', zero_division=0)

top3_acc = np.nan
macro_auc = np.nan
y_test_bin = None
if y_test_proba is not None:
    all_labels = np.arange(n_classes)
    top3_acc = top_k_accuracy_score(y_test, y_test_proba, k=3, labels=all_labels)
    y_test_bin = label_binarize(y_test, classes=all_labels)      # one-vs-rest indicator matrix
    try:
        macro_auc = roc_auc_score(y_test_bin, y_test_proba, average='macro')
    except ValueError as e:
        print(f'Macro ROC-AUC could not be computed: {e}')

overall_metrics = pd.DataFrame({
    'metric': ['Accuracy', 'Top-3 accuracy',
               'Precision (macro)', 'Recall (macro)', 'F1-score (macro)',
               'ROC-AUC (macro, one-vs-rest)'],
    'value': [accuracy_score(y_test, y_test_pred), top3_acc,
              prec_m, rec_m, f1_m, macro_auc],
})

print(f'\nOverall test-set metrics - {winner_name}:')
display(overall_metrics.round(4))
overall_metrics.to_csv(TABLE_DIR / 'overall_test_metrics.csv', index=False)

In [ ]:
# ---------------------------------------------------------------- per-class F1-score chart
per_class_f1 = report_df.loc[list(le.classes_)].sort_values('f1-score')

plt.figure(figsize=(9, 8))
plt.barh(per_class_f1.index, per_class_f1['f1-score'], color='#4C72B0')
plt.axvline(f1_m, color='crimson', linestyle='--', label=f'Macro-average F1 = {f1_m:.3f}')
plt.xlabel('F1-score')
plt.xlim(0, 1.05)
plt.title(f'Per-Class F1-Score on the Test Set - {winner_name}')
plt.grid(axis='y', visible=False)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'per_class_f1.png')
plt.show()

In [ ]:
# ---------------------------------------------------------------- confusion matrix (raw counts)
cm = confusion_matrix(y_test, y_test_pred)
cm_table = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
cm_table.to_csv(TABLE_DIR / 'confusion_matrix_counts.csv')

plt.figure(figsize=(14, 12))
ax = sns.heatmap(cm, mask=(cm == 0), annot=True, fmt='d', cmap='Blues', vmin=0, vmax=cm.max(),
                 annot_kws={'size': 7}, cbar_kws={'label': 'Number of images', 'ticks': np.arange(0, cm.max() + 1)},
                 xticklabels=le.classes_, yticklabels=le.classes_, linewidths=0.3, linecolor='lightgray')
ax.grid(False)
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.title(f'Confusion Matrix (counts) - {winner_name}, test set')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix.png')
plt.show()

In [ ]:
# ---------------------------------------------------------------- ROC curves and per-class ROC-AUC
if y_test_proba is not None:
    fpr_grid = np.linspace(0, 1, 200)      # common false-positive-rate axis for averaging
    tpr_interpolated = []

    plt.figure(figsize=(8, 7))
    for i, cname in enumerate(le.classes_):
        if y_test_bin[:, i].sum() == 0:    # class absent from the test set: no ROC curve
            continue
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_test_proba[:, i])
        plt.plot(fpr, tpr, color='steelblue', alpha=0.35, linewidth=1)
        tpr_interpolated.append(np.interp(fpr_grid, fpr, tpr))

    mean_tpr = np.mean(tpr_interpolated, axis=0)                  # macro-average curve
    plt.plot([], [], color='steelblue', alpha=0.5, label='Individual classes')
    plt.plot(fpr_grid, mean_tpr, color='navy', linewidth=2.5,
             label=f'Macro-average (AUC = {macro_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Chance')
    plt.xlim(0, 1)
    plt.ylim(0, 1.02)
    plt.xlabel('False positive rate')
    plt.ylabel('True positive rate')
    plt.title(f'One-vs-Rest ROC Curves - {winner_name}, test set')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'roc_curves.png')
    plt.show()

    # per-class ROC-AUC table
    per_class_auc = {}
    for i, cname in enumerate(le.classes_):
        try:
            per_class_auc[cname] = roc_auc_score(y_test_bin[:, i], y_test_proba[:, i])
        except ValueError:
            per_class_auc[cname] = float('nan')     # too few samples for this class

    auc_df = pd.DataFrame.from_dict(per_class_auc, orient='index', columns=['ROC_AUC'])
    auc_df = auc_df.join(report_df[['support']])
    auc_df = auc_df.sort_values('ROC_AUC', ascending=False)

    print('Per-class ROC-AUC (NaN = too few test samples):')
    display(auc_df.round(4))
    auc_df.to_csv(TABLE_DIR / 'per_class_roc_auc.csv')
else:
    print(f'{winner_name} does not provide probability estimates - ROC analysis skipped.')

## 9. Error Analysis

Every test image that the selected model classified incorrectly is listed with its true class, its predicted class and the confidence of the prediction. Because filenames are numeric serial numbers that are unique only within a class folder, each image is identified by its relative path (`test/<class>/<filename>`).

The section further reports the most frequent confusions between pairs of classes, compares the confidence distribution of correct and incorrect predictions, and shows a gallery of all misclassified test images.

In [ ]:
pred_labels = le.inverse_transform(y_test_pred)
true_labels_arr = le.inverse_transform(y_test)
if y_test_proba is not None:
    confidences = y_test_proba.max(axis=1)              # probability of the predicted class
else:
    confidences = np.full(len(y_test_pred), np.nan)

mis_idx = np.where(y_test_pred != y_test)[0]            # positions of the misclassified test images

mis_df = pd.DataFrame({
    'image_path': [test_paths[i].relative_to(SPLIT_DIR).as_posix() for i in mis_idx],
    'filename': [test_files[i] for i in mis_idx],
    'true_label': [true_labels_arr[i] for i in mis_idx],
    'predicted_label': [pred_labels[i] for i in mis_idx],
    'confidence': [confidences[i] for i in mis_idx],
}).sort_values(['true_label', 'filename']).reset_index(drop=True)

print(f'{len(mis_df)} of {len(y_test)} test images were misclassified ({len(mis_df) / len(y_test) * 100:.1f}%).\n')
display(mis_df.round(4))
mis_df.to_csv(TABLE_DIR / 'misclassified_test_images.csv', index=False)

In [ ]:
# ---------------------------------------------------------------- most frequent class confusions
confusion_pairs = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append([le.classes_[i], le.classes_[j], int(cm[i, j])])

pairs_df = pd.DataFrame(confusion_pairs, columns=['true_label', 'predicted_label', 'count'])
pairs_df = pairs_df.sort_values('count', ascending=False).reset_index(drop=True)

print('Most frequent confusions (true class -> predicted class):')
display(pairs_df.head(15))
pairs_df.to_csv(TABLE_DIR / 'top_confused_class_pairs.csv', index=False)   # full list of confusions

# ---------------------------------------------------------------- confidence of correct vs. incorrect predictions
if y_test_proba is not None:
    is_correct = (y_test_pred == y_test)
    confidence_df = pd.DataFrame({
        'confidence': confidences,
        'outcome': np.where(is_correct, 'Correct', 'Misclassified'),
    })
    # mean confidence of each group (NaN if the group is empty, e.g. no misclassified images)
    mean_conf_correct = confidences[is_correct].mean() if is_correct.any() else np.nan
    mean_conf_wrong = confidences[~is_correct].mean() if (~is_correct).any() else np.nan
    print(f'\nMean confidence - correct: {mean_conf_correct:.3f}, misclassified: {mean_conf_wrong:.3f}')

    plt.figure(figsize=(8, 5))
    sns.histplot(data=confidence_df, x='confidence', hue='outcome', bins=10, binrange=(0, 1),
                 multiple='dodge', palette={'Correct': '#55A868', 'Misclassified': '#C44E52'}, shrink=0.85)
    plt.xlabel('Prediction confidence (probability of the predicted class)')
    plt.ylabel('Number of test images')
    plt.title(f'Confidence Distribution - {winner_name}, test set')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'confidence_distribution.png')
    plt.show()

In [ ]:
# ---------------------------------------------------------------- gallery of all misclassified test images
n_cols = 4
total = len(mis_idx)
num_pages = math.ceil(total / MISCLASSIFIED_PER_PAGE)

if total == 0:
    print('No misclassified test images - nothing to plot.')
else:
    # one pass of the loop produces one figure (page) of the gallery
    for page in range(num_pages):
        start = page * MISCLASSIFIED_PER_PAGE                 # first image on this page
        end = min(start + MISCLASSIFIED_PER_PAGE, total)      # the last page may be shorter
        page_idx = mis_idx[start:end]

        n_rows = math.ceil(len(page_idx) / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
        axes = np.array(axes).reshape(-1)

        for ax_i, idx in enumerate(page_idx):
            axes[ax_i].imshow(load_and_orient_image(test_paths[idx]))
            axes[ax_i].axis('off')
            if np.isnan(confidences[idx]):
                conf_str = 'n/a'
            else:
                conf_str = f'{confidences[idx] * 100:.0f}%'
            axes[ax_i].set_title(
                f'{test_files[idx]}\nTrue: {true_labels_arr[idx]}\nPred: {pred_labels[idx]} ({conf_str})',
                fontsize=9, color='crimson')

        # hide unused axes on the last page
        for ax_i in range(len(page_idx), len(axes)):
            axes[ax_i].axis('off')

        plt.suptitle(f'Misclassified Test Images {start + 1}-{end} of {total} (page {page + 1}/{num_pages})',
                     fontsize=13)
        plt.tight_layout()
        plt.savefig(FIG_DIR / f'misclassified_test_images_page{page + 1}.png')
        plt.show()

    print(f'All {total} misclassified images are shown on {num_pages} page(s).')

## 10. Inference Demonstration

This section demonstrates the deployed pipeline on a single image: Xception feature extraction, the saved PCA projection, and the selected classifier. By default it classifies the first *gulgula* image of the hand-selected test set. To classify your own photo, upload it to Google Drive and set `QUERY_IMAGE_PATH` to its path (for example `DRIVE_DATA_ROOT / 'my_pitha.jpg'`). If its true class is known, set `QUERY_IMAGE_TRUE_LABEL` to the exact class folder name; a wrong prediction is then displayed next to a reference image of the true class and one of the predicted class for visual comparison. Leave it as `None` when the true class is unknown.

The prediction is accompanied by the class metadata stored in `pitha_metadata.json` (region, cooking method, ingredients, nutritional values), and the three most probable classes are tabulated. If the image file does not exist, the demonstration is skipped.

In [ ]:
QUERY_IMAGE_PATH = None           # None = use a test image; or your own photo, e.g. DRIVE_DATA_ROOT / 'my_pitha.jpg'
QUERY_IMAGE_TRUE_LABEL = None     # true class name of your own photo, or None if unknown

if QUERY_IMAGE_PATH is None:
    # default: the first test image of 'gulgula' (or of the first class), whose true class is known
    demo_class = 'gulgula' if 'gulgula' in CLASS_NAMES else CLASS_NAMES[0]
    QUERY_IMAGE_PATH = sample_image_for_class(demo_class, split='test')
    QUERY_IMAGE_TRUE_LABEL = demo_class
QUERY_IMAGE_PATH = Path(QUERY_IMAGE_PATH) if QUERY_IMAGE_PATH is not None else None

# Class metadata (optional)
metadata_path = SCRIPT_DIR / 'pitha_metadata.json'
if metadata_path.exists():
    with open(metadata_path, encoding='utf-8') as f:
        METADATA = json.load(f)
else:
    METADATA = {}
    print('pitha_metadata.json not found - only the class label will be reported.')

def classify_one(image_path):
    # Runs the full pipeline on one image; returns (predicted label, confidence, image, class probabilities)
    pil_img = load_and_orient_image(image_path)
    padded = resize_with_pad(np.array(pil_img), size=TARGET_SIZE).astype(np.float32)
    arr = _preprocess(np.expand_dims(padded, axis=0))
    feat_raw = np.asarray(extractor(arr, training=False))     # Xception features
    feat_pca = pca.transform(feat_raw)                        # saved PCA projection
    pred_idx = winner.predict(feat_pca)[0]
    pred_label = le.inverse_transform([pred_idx])[0]
    if hasattr(winner, 'predict_proba'):
        proba = winner.predict_proba(feat_pca)[0]
        confidence = float(proba[pred_idx])
    else:
        proba = None
        confidence = None
    return pred_label, confidence, pil_img, proba

run_inference = QUERY_IMAGE_PATH is not None and QUERY_IMAGE_PATH.exists()

if not run_inference:
    print(f'Query image not found: {QUERY_IMAGE_PATH}')
    print('Set QUERY_IMAGE_PATH to an existing image to run the demonstration.')
else:
    pred_label, confidence, query_img, proba = classify_one(QUERY_IMAGE_PATH)
    meta = METADATA.get(pred_label, {})

    print(f"Predicted pitha type : {meta.get('display_name', pred_label)}")
    if confidence is not None:
        print(f'Confidence           : {confidence * 100:.1f}%')
    print(f"Division / District  : {meta.get('division', '?')} / {meta.get('district', '?')}")
    print(f"Cooking method       : {meta.get('cooking_method', '?')}")
    if meta.get('key_ingredients'):
        print(f"Key ingredients      : {', '.join(meta['key_ingredients'])}")
    print(f"Recipe summary       : {meta.get('recipe_summary', '?')}")

    # Nutritional values (per serving as given in the metadata file)
    print('Nutritional values:')
    for key, label in [('calories', 'Calories'), ('protein_g', 'Protein (g)'),
                       ('carbs_g', 'Carbohydrates (g)'), ('fat_g', 'Fat (g)'), ('sugar_g', 'Sugar (g)')]:
        if key in meta:
            print(f'  {label:18}: {meta[key]}')

    # ---------------------------------------------------------------- result tables
    inference_result = pd.DataFrame([{
        'image': QUERY_IMAGE_PATH.name,
        'predicted_class': pred_label,
        'confidence': confidence,
        'division': meta.get('division'),
        'district': meta.get('district'),
        'cooking_method': meta.get('cooking_method'),
        'key_ingredients': ', '.join(meta.get('key_ingredients', [])),
    }])
    inference_result.to_csv(TABLE_DIR / 'inference_result.csv', index=False, encoding='utf-8-sig')

    if proba is not None:
        top3_idx = np.argsort(proba)[::-1][:3]          # indices of the three most probable classes
        top3_df = pd.DataFrame({'rank': [1, 2, 3],
                                'class': le.classes_[top3_idx],
                                'probability': proba[top3_idx]})
        print('\nTop-3 predictions:')
        display(top3_df.round(4))
        top3_df.to_csv(TABLE_DIR / 'inference_top3.csv', index=False)

In [ ]:
if run_inference:
    if QUERY_IMAGE_TRUE_LABEL is None or QUERY_IMAGE_TRUE_LABEL == pred_label:
        # a single panel: the query image with its prediction
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(query_img)
        ax.axis('off')
        if QUERY_IMAGE_TRUE_LABEL is None:
            ax.set_title(f'Predicted: {pred_label}', fontsize=12)
        else:
            ax.set_title(f'Correct prediction: {pred_label}', fontsize=12, color='green')
    else:
        # misclassified: query image beside a reference image of the true class and of the predicted class
        true_ref_path = sample_image_for_class(QUERY_IMAGE_TRUE_LABEL)
        pred_ref_path = sample_image_for_class(pred_label)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(query_img)
        axes[0].set_title('Query image', fontsize=11)

        if true_ref_path is not None:
            axes[1].imshow(load_and_orient_image(true_ref_path))
            axes[1].set_title(f'True class:\n{QUERY_IMAGE_TRUE_LABEL}', fontsize=11, color='green')
        else:
            axes[1].text(0.5, 0.5, 'no reference\nimage found', ha='center', va='center')

        if pred_ref_path is not None:
            axes[2].imshow(load_and_orient_image(pred_ref_path))
            axes[2].set_title(f'Predicted class:\n{pred_label}', fontsize=11, color='crimson')
        else:
            axes[2].text(0.5, 0.5, 'no reference\nimage found', ha='center', va='center')

        for ax in axes:
            ax.axis('off')
        plt.suptitle('Misclassified image - visual comparison', fontsize=13)
        print(f'Predicted "{pred_label}", true class "{QUERY_IMAGE_TRUE_LABEL}" - misclassified.')

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'inference_demo.png')
    plt.show()

## 11. Output Summary

### Model artifacts (`model_xception/`)

| File | Contents |
|---|---|
| `pitha_pca_xception.pkl` | Fitted PCA transform (training data only) |
| `pitha_features_xception.npz` | PCA features, labels and filenames for the training, validation and test sets |
| `pitha_backbone_xception.json` | Backbone name, raw feature dimension, PCA components, retained variance |
| `pitha_label_encoder.pkl` | Class name ↔ integer mapping shared by all classifiers |
| `pitha_model_xception.pkl` | Selected classifier (joblib) |
| `pitha_model_xception.info.json` | Name of the selected classifier with its validation and test metrics |
| `split_manifest.json` | Exact file-to-partition assignment of the split and every removal made while cleaning |
| `run_info.json` | Data locations, split fingerprint, image counts and headline result of this run |

### Tables (`model_xception/tables/`)

| File | Contents |
|---|---|
| `experimental_setup.csv`, `software_environment.csv` | Experiment configuration and software versions |
| `split_summary.csv`, `dataset_summary_by_class.csv` | Subset sizes and per-class image counts |
| `feature_matrix_shapes.csv`, `pca_explained_variance.csv` | Feature matrix dimensions and PCA variance per component |
| `classifier_comparison.csv` | Validation accuracy, test accuracy and macro F1 of all four classifiers |
| `overall_test_metrics.csv`, `per_class_metrics.csv` | Overall and per-class test metrics of the selected model |
| `confusion_matrix_counts.csv`, `per_class_roc_auc.csv` | Confusion matrix and per-class ROC-AUC |
| `misclassified_test_images.csv`, `top_confused_class_pairs.csv` | Every wrong prediction and the most frequent class confusions |
| `inference_result.csv`, `inference_top3.csv` | Result of the single-image inference demonstration |
| `raw_input_counts.csv` | Images per class as collected: `clean_data/` pool and hand-selected test folder |
| `data_cleaning_summary.csv` | Collected, removed (test overlap, duplicates, unreadable) and used image counts |
| `split_fingerprint.csv` | Fingerprint of the exact split, to confirm all notebooks used the same partition |

### Figures (`model_xception/figures/`)

| File | Contents |
|---|---|
| `class_distribution.png`, `sample_images_per_class.png`, `augmentation_examples.png` | Dataset overview and augmentation examples |
| `classifier_accuracy_comparison.png` | Validation and test accuracy of all classifiers |
| `per_class_f1.png` | Per-class F1-score of the selected model |
| `confusion_matrix.png` | Confusion matrix (image counts) |
| `roc_curves.png` | One-vs-rest ROC curves with the macro-average |
| `confidence_distribution.png` | Prediction confidence of correct and incorrect predictions |
| `misclassified_test_images_page*.png` | Gallery of all misclassified test images |
| `inference_demo.png` | Single-image inference demonstration |

### Reporting notes

- The **test accuracy** of the selected model is the headline generalization result. It was never consulted during model selection.
- The **validation accuracy** documents the selection criterion and should not be reported as a final result.
- **Macro-averaged** metrics weight every class equally, whatever its size.

In [ ]:
# List every file generated by this run, so the outputs can be checked before submission
print('Model artifacts  :', MODEL_DIR)
for f in sorted(MODEL_DIR.iterdir()):
    if f.is_file():
        print('   ', f.name)

print('\nTables           :', TABLE_DIR)
for f in sorted(TABLE_DIR.glob('*.csv')):
    print('   ', f.name)

print('\nFigures          :', FIG_DIR)
for f in sorted(FIG_DIR.glob('*.png')):
    print('   ', f.name)

## 12. Download the Results

Everything this run produced — trained model, PCA transform, label encoder, extracted features, the split manifest, every figure (`figures/`) and every table (`tables/`) — is packed into one zip file. The zip is downloaded to your computer, and a copy is also saved to `pitha_outputs/` on Google Drive, so the results are not lost if the browser blocks the download or the Colab session disconnects.

In [ ]:
import shutil
from datetime import datetime

# ---------------------------------------------------------------- run record, saved inside the zip
run_info = {
    'backbone': 'Xception',
    'finished_at': datetime.now().isoformat(timespec='seconds'),
    'train_val_pool': str(CURATED_DIR),
    'test_set': str(TEST_INPUT_DIR),
    'split_fingerprint': SPLIT_FINGERPRINT,
    'images_train_original': int(train_orig),
    'images_train_augmented': int(train_aug),
    'images_validation': int(val_total),
    'images_test': int(test_total),
    'selected_model': winner_name,
    'selected_model_test_accuracy_pct': round(float(results[winner_name]['test_acc']) * 100, 2),
    'best_test_accuracy_model': str(results_df['test_acc'].astype(float).idxmax()),
    'best_test_accuracy_pct': round(float(results_df['test_acc'].astype(float).max()) * 100, 2),
    'compute_device': 'GPU' if gpus else 'CPU',
}
with open(MODEL_DIR / 'run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2)

# ---------------------------------------------------------------- zip the whole output folder
zip_path = Path(shutil.make_archive(str(MODEL_DIR.parent / f'{MODEL_DIR.name}_results'), 'zip',
                                    root_dir=MODEL_DIR.parent, base_dir=MODEL_DIR.name))
n_files = sum(1 for f in MODEL_DIR.rglob('*') if f.is_file())
print(f'Created {zip_path.name}: {n_files} files, {zip_path.stat().st_size / 1e6:.1f} MB')

# ---------------------------------------------------------------- keep a copy on Google Drive
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, DRIVE_OUTPUT_DIR / zip_path.name)
print(f'Copy saved to Drive : {DRIVE_OUTPUT_DIR / zip_path.name}')

# ---------------------------------------------------------------- download to this computer
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print(f'Download the zip from: {zip_path}')